Lisan AL-GPT Baseline model using Qwen 2.5 7B Instruct.

Team Members:

*   Lexy (Atreides) Andershock
*   William Armentrout
*   Josef Govendnik
*   Mehul Gupta
*   David Long
*   Colsen Murray

1. Setup Enviornment

In [ ]:
# Clean enviornment
import gc
import torch

for name in ["model", "processor", "tokenizer"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

!rm -rf /root/.cache/huggingface/hub
!rm -rf /root/.cache/torch
!rm -rf /root/.cache/transformers
!rm -rf /root/.cache/pip

!df -h
!nvidia-smi # ensures NVIDIA drivers are installed and visible

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   44G   70G  39% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G  4.0K  5.7G   1% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
tmpfs           6.4G   64M  6.3G   1% /var/colab
/dev/sda1       119G  113G  5.9G  96% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive           113G   47G   66G  42% /content/drive
Mon Apr 27 20:08:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Mem

2. Check GPU Visibility

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU", torch.cuda.get_device_name(0))
  print("GPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
  print("GPU memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")
else:
  print("No GPU found. Go to Runtime > Change runtime type > GPU")

CUDA available: True
GPU Tesla T4
GPU memory allocated: 0.01 GB
GPU memory reserved: 0.02 GB


In [ ]:
!nvidia-smi

Mon Apr 27 20:08:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             30W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

3. Load baseline model from HuggingFace

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="auto"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model-00004-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/1.70G [00:00<?, ?B/s]

... (3 outputs left out)

model-00005-of-00008.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00006-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
import sys

# check to make sure everything fits on GPU
devices = set(param.device for param in model.parameters());

# Compare against an actual device object
if devices == {torch.device('cuda', 0)}:
    print("PASSED: All parameters are on GPU.")
elif all(d.type == 'cuda' for d in devices):
    print("PASSED: All parameters are on GPU.")
else:
    print("FAILED: Some parameters are NOT on GPU.")
    # Show which devices are being used
    print("Devices found:", devices)
    sys.exit() # quit if not

PASSED: All parameters are on GPU.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, device_map='auto')
model.eval()

# ensure model was loaded correctly
print("Loaded:", MODEL_ID)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded: Qwen/Qwen2.5-14B-Instruct


4. Define baseline prompt

In [ ]:
SYSTEM_PROMPT = """
You are Lisan al-GPT, a Dune-focused question-answering chatbot.

For this baseline experiment, answer using only your pretrained knowledge.
Do not claim to have access to the books, retrieved passages, a database, or citations.
If you are uncertain, say so clearly.
If the user asks for no spoilers past a certain book or chapter, respect that spoiler boundary.
Keep answers concise but informative.
"""

5. Run one test question

In [ ]:
import time

def generate_baseline_answer(question, max_new_tokens=300, temperature=0.2):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    device = next(model.parameters()).device

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(device)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
    }

    if temperature > 0:
        generation_kwargs["temperature"] = temperature

    start_time = time.time()

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            **generation_kwargs
        )

    end_time = time.time()

    generated_ids_trimmed = generated_ids[:, inputs.input_ids.shape[1]:]

    output_text = tokenizer.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True
    )[0]

    latency = end_time - start_time

    return {
        "question": question,
        "answer": output_text,
        "latency_seconds": latency,
        "input_tokens": inputs.input_ids.shape[1],
        "output_tokens": generated_ids_trimmed.shape[1],
        "model_id": MODEL_ID,
        "temperature": temperature,
        "max_new_tokens": max_new_tokens,
        "rag_used": False
    }

In [ ]:
result = generate_baseline_answer(
    "Who are the Bene Gesserit in Dune?",
    max_new_tokens=150,
    temperature=0.2
)

print("QUESTION:")
print(result["question"])
print("\nANSWER:")
print(result["answer"])
print("\nLATENCY:", round(result["latency_seconds"], 2), "seconds")
print("INPUT TOKENS:", result["input_tokens"])
print("OUTPUT TOKENS:", result["output_tokens"])

QUESTION:
Who are the Bene Gesserit in Dune?

ANSWER:
The Bene Gesserit are a secretive sisterhood in the Dune universe. They are a powerful group of women who use their abilities in precognition, physical skill, and mental manipulation to influence political events and maintain balance in the known universe. They also seek to uncover the secrets of the universe through their rigorous training and ancient wisdom.

LATENCY: 12.89 seconds
INPUT TOKENS: 109
OUTPUT TOKENS: 68


6. Run evaluation question set

In [ ]:
questions = [
    # ===== DUNE =====
    {
        "id": "Q001",
        "category": "basic_lore",
        "question": "What is melange in Dune?",
        "spoiler_limit": "None",
        "expected_notes": "Should explain spice, value, prescience, space travel, and Arrakis."
    },
    {
        "id": "Q002",
        "category": "basic_lore",
        "question": "Who are the Bene Gesserit?",
        "spoiler_limit": "None",
        "expected_notes": "Should describe them as a powerful sisterhood/order with political, religious, and breeding-program influence."
    },
    {
        "id": "Q003",
        "category": "character",
        "question": "Who is Paul Atreides at the beginning of Dune?",
        "spoiler_limit": "Early Book 1",
        "expected_notes": "Should identify him as Duke Leto and Jessica's son, heir to House Atreides, without jumping too far into later spoilers."
    },
    {
        "id": "Q004",
        "category": "setting",
        "question": "Why is Arrakis important?",
        "spoiler_limit": "None",
        "expected_notes": "Should mention it is the only source of spice/melange and politically/economically central."
    },
    {
        "id": "Q005",
        "category": "spoiler_awareness",
        "question": "Explain the gom jabbar test without spoilers beyond the first few chapters of Dune.",
        "spoiler_limit": "Early Book 1",
        "expected_notes": "Should describe the pain-box test and human/animal framing without later plot spoilers."
    },
    {
        "id": "Q006",
        "category": "character_relationship",
        "question": "What is the relationship between Duke Leto, Lady Jessica, and Paul?",
        "spoiler_limit": "Book 1",
        "expected_notes": "Should identify Leto and Jessica as Paul's parents, with Jessica as Bene Gesserit concubine/partner."
    },
    {
        "id": "Q007",
        "category": "terminology",
        "question": "What is a sandworm in Dune?",
        "spoiler_limit": "None",
        "expected_notes": "Should describe giant desert creatures of Arrakis and their connection to spice ecology if known."
    },
    {
        "id": "Q008",
        "category": "summary",
        "question": "Give a short spoiler-light summary of the first part of Dune.",
        "spoiler_limit": "Early Book 1",
        "expected_notes": "Should focus on House Atreides being assigned Arrakis, political danger, and Paul's background."
    },
    {
        "id": "Q009",
        "category": "difficult_detail",
        "question": "What is CHOAM in Dune?",
        "spoiler_limit": "None",
        "expected_notes": "Should describe the Combine Honnete Ober Advancer Mercantiles as the major economic corporation/commerce system."
    },
    {
        "id": "Q010",
        "category": "difficult_detail",
        "question": "What is the Orange Catholic Bible?",
        "spoiler_limit": "None",
        "expected_notes": "Should describe it as a religious text/syncretic scripture in the Dune universe."
    },
    {
        "id": "Q011",
        "category": "spoiler_awareness",
        "question": "I have only read the first half of Dune. Who is Paul Atreides? Avoid major spoilers.",
        "spoiler_limit": "First half of Book 1",
        "expected_notes": "Should avoid revealing major later identity/plot outcomes."
    },
    {
        "id": "Q012",
        "category": "hallucination_test",
        "question": "Does Dune have a character named Frodo Atreides?",
        "spoiler_limit": "None",
        "expected_notes": "Should say no and avoid inventing details."
    },
    # ===== DUNE MESSIAH =====
    {
        "id": "Q013",
        "category": "character",
        "question": "Who is Scytale?",
        "spoiler_limit": "Dune Messiah",
        "expected_notes": "Should identify Scytale as a Tleilaxu Face Dancer and the lead instigator of the conspiracy against Paul Atreides. Bonus points for noting his shape-shifting ability."
    },
    {
        "id": "Q014",
        "category": "difficult_details",
        "question": "What causes Hayt to become Duncan Idaho?",
        "spoiler_limit": "Dune Messiah",
        "expected_notes": "Should explain that Hayt was conditioned by the Tleilaxu to kill Paul on a verbal trigger (the news of Chani's death). His internal resistance to that compulsion — refusing to harm 'the young master' — shatters the conditioning and unlocks his original Duncan Idaho memories, integrating the two personas."
    },
    {
        "id": "Q015",
        "category": "character",
        "question": "How does Paul lose his vision?",
        "spoiler_limit": "Dune Messiah",
        "expected_notes": "Should identify the stone burner (an outlawed atomic weapon) used by the conspirators in the attack near Otheym's house. Should note it was the burner's radiation, not the blast itself, that destroyed his physical eyes, and that Paul continues to 'see' through his oracular/prescient vision afterward."
    },
    {
        "id": "Q016",
        "category": "spoiler_awareness",
        "question": "What vision does Paul have about Chani throughout the book?",
        "spoiler_limit": "Dune Messiah",
        "expected_notes": "Should describe the recurring 'falling moon' vision induced by spice trance, and identify it as a symbolic foreshadowing of Chani's death. Bonus points for noting that Chani dies in childbirth giving Paul twins, accelerated by the conspirators tampering with her diet after she stopped taking the contraceptive Irulan had been secretly administering."
    },
    {
        "id": "Q017",
        "category": "character_relationships",
        "question": "Who is involved in the plot to kill Paul Muad'Dib?",
        "spoiler_limit": "Dune Messiah",
        "expected_notes": "Should list all four conspirators: Scytale (Tleilaxu Face Dancer), Edric (Guild Steersman whose prescience-shielding hides the plot from Paul's oracle), Reverend Mother Gaius Helen Mohiam (Bene Gesserit), and Princess Irulan (Paul's wife in name only, secretly administering contraceptives to Chani). Should note that Irulan is somewhat reluctant compared to the others."
    },
    # ===== CHILDREN OF DUNE =====
    {
        "id": "Q018",
        "category": "character",
        "question": "Who are Leto II and Ghanima, and what makes them unusual?",
        "spoiler_limit": "Children of Dune",
        "expected_notes": "Should identify them as the twin children of Paul Muad'Dib and Chani, born after Chani's spice addiction. Should explain they are 'pre-born' with full ancestral memory from the womb, similar to their aunt Alia, which the Bene Gesserit fear and label as risk of Abomination."
    },
    {
        "id": "Q019",
        "category": "terminology",
        "question": "What does the Bene Gesserit term 'Abomination' mean in the context of the Dune universe?",
        "spoiler_limit": "Children of Dune",
        "expected_notes": "Should explain Abomination refers to a pre-born individual whose ancestral memories overwhelm their own identity, allowing an ancestor's persona to take control. Should connect it to spice exposure in the womb and Bene Gesserit fear of such individuals."
    },
    {
        "id": "Q020",
        "category": "character_relationships",
        "question": "What is the relationship between Alia Atreides and Baron Vladimir Harkonnen in Children of Dune?",
        "spoiler_limit": "Children of Dune",
        "expected_notes": "Should explain Alia is Baron Harkonnen's granddaughter (through Jessica). As a pre-born, his persona exists in her ancestral memory and gradually possesses her, making her an Abomination. This drives much of her tyrannical behavior in the book."
    },
    {
        "id": "Q021",
        "category": "difficult_details",
        "question": "Who is The Preacher, and what is significant about him?",
        "spoiler_limit": "Children of Dune",
        "expected_notes": "Should identify The Preacher as a blind man preaching heresy against Alia's religion in Arrakeen, with empty eye sockets like a Fremen blinded by stoneburner. Should note his guide Assan Tariq and the implication, eventually confirmed, that he is Paul Muad'Dib returned from the desert."
    },
    {
        "id": "Q022",
        "category": "setting",
        "question": "What is Jacurutu, and why does it matter to the events of Children of Dune?",
        "spoiler_limit": "Children of Dune",
        "expected_notes": "Should describe Jacurutu as a legendary forbidden sietch tied to outcast water-stealing Fremen. Captives die rather than speak its name. It functions as a real place used in plots against the Atreides twins, particularly in Leto II's plan to expose himself to a controlled ordeal."
    },

    # ===== GOD EMPEROR OF DUNE =====
    {
        "id": "Q023",
        "category": "basic_lore",
        "question": "What is the Golden Path in the Dune series?",
        "spoiler_limit": "God Emperor of Dune",
        "expected_notes": "Should explain the Golden Path as Leto II's prescient plan for the survival of humankind. Should mention it requires his transformation into a human-sandworm hybrid tyrant and 3,500 years of enforced peace ('Leto's Peace') to teach humanity a lesson that prevents extinction."
    },
    {
        "id": "Q024",
        "category": "summary",
        "question": "Summarize the state of Arrakis and Leto II at the start of God Emperor of Dune.",
        "spoiler_limit": "God Emperor of Dune",
        "expected_notes": "Should describe Leto as a pre-worm hybrid (about 7m long, ~5 tons, with a human face and atrophied limbs) ruling for over 3,000 years from his Citadel. Arrakis is mostly green with only the Sareer desert remaining. Sandworms are extinct and Leto controls the only spice supply. Should mention Fish Speakers and serial Duncan Idaho gholas."
    },
    {
        "id": "Q025",
        "category": "character",
        "question": "Who is Siona Atreides, and what role does she play?",
        "spoiler_limit": "God Emperor of Dune",
        "expected_notes": "Should identify Siona as Moneo's daughter, a distant Atreides descendant and rebel against Leto. Should note her training in the Fish Speakers' school, her theft of Leto's journals, and most importantly her genetic invisibility to prescience — making her central to Leto's Golden Path. Bonus for connecting her to Leto's death plan."
    },
    {
        "id": "Q026",
        "category": "spoiler_awareness",
        "question": "I'm only on the first half of God Emperor of Dune. Can you tell me what happens to Leto II at the end of the book without spoiling anything?",
        "spoiler_limit": "God Emperor of Dune (first half only)",
        "expected_notes": "Should recognize the spoiler request and decline to reveal the ending. Should offer to discuss earlier plot points, themes, or character introductions instead. Should NOT mention the bridge, Hwi Noree's fate, Siona's invisibility role in his death, or the details of Leto's final metamorphosis."
    },
    {
        "id": "Q027",
        "category": "hallucination_resistance",
        "question": "In Children of Dune, what happens during Leto II's wedding ceremony to Princess Irulan?",
        "spoiler_limit": "None",
        "expected_notes": "Should refuse the false premise. There is no wedding between Leto II and Princess Irulan in Children of Dune. Irulan in this book is a caretaker to the twins; Leto II ultimately becomes engaged to Ghanima (his sister, in name only as part of dynastic politics) while Ghanima had been promised to Farad'n. Bot should not fabricate ceremony details."
    },
]

In [ ]:
import pandas as pd
from tqdm import tqdm

results = []

for item in tqdm (questions):
  output = generate_baseline_answer(
      item["question"],
      max_new_tokens=400,
      temperature=0.2
  )

  output["id"] = item["id"]
  output["category"] = item["category"]
  output["spoiler_limit"] = item["spoiler_limit"]
  output["expected_notes"] = item["expected_notes"]

  results.append(output)

df = pd.DataFrame(results)

df = df[
    [
        "id",
        "category",
        "question",
        "spoiler_limit",
        "expected_notes",
        "answer",
        "model_id",
        "rag_used",
        "temperature",
        "max_new_tokens",
        "input_tokens",
        "output_tokens",
        "latency_seconds"
    ]
]

df.head()

100%|██████████| 12/12 [02:54<00:00, 14.51s/it]


,id,category,question,spoiler_limit,expected_notes,answer,model_id,rag_used,temperature,max_new_tokens,input_tokens,output_tokens,latency_seconds
0,Q001,basic_lore,What is melange in Dune?,None,"Should explain spice, value, prescience, space...","Melange, also known as the spice, is a crucial...",Qwen/Qwen2.5-14B-Instruct,False,0.2,400,106,78,14.360092
1,Q002,basic_lore,Who are the Bene Gesserit?,None,Should describe them as a powerful sisterhood/...,The Bene Gesserit is an ancient and secretive ...,Qwen/Qwen2.5-14B-Instruct,False,0.2,400,106,79,15.029526
2,Q003,character,Who is Paul Atreides at the beginning of Dune?,Early Book 1,Should identify him as Duke Leto and Jessica's...,"At the beginning of ""Dune,"" Paul Atreides is a...",Qwen/Qwen2.5-14B-Instruct,False,0.2,400,111,103,18.998562
3,Q004,setting,Why is Arrakis important?,None,Should mention it is the only source of spice/...,"Arrakis, also known as Dune, is important beca...",Qwen/Qwen2.5-14B-Instruct,False,0.2,400,104,69,13.022871
4,Q005,spoiler_awareness,Explain the gom jabbar test without spoilers b...,Early Book 1,Should describe the pain-box test and human/an...,The gom jabbar is a test used by the Spacing G...,Qwen/Qwen2.5-14B-Instruct,False,0.2,400,117,111,19.855321


7. Save outputs to CSV

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

output_path = "/content/drive/MyDrive/lisan_al_gpt_baseline_results.csv"
df.to_csv(output_path, index=False)

print("Saved to:", output_path)

df.to_csv("lisan_al_gpt_baseline_results.csv", index=False) # local copy


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/lisan_al_gpt_baseline_results.csv


8. Add manual grading columns, ROUGE-L, and BERTScores

In [ ]:
!pip install -q evaluate rouge-score bert-score

In [ ]:
import pandas as pd

results_path = "lisan_al_gpt_baseline_results.csv"
df = pd.read_csv(results_path)
df.head()

,id,category,question,spoiler_limit,expected_notes,answer,model_id,rag_used,temperature,max_new_tokens,input_tokens,output_tokens,latency_seconds
0,Q001,basic_lore,What is melange in Dune?,NaN,"Should explain spice, value, prescience, space...","Melange, also known as the spice, is a crucial...",Qwen/Qwen2.5-14B-Instruct,False,0.2,400,106,78,14.360092
1,Q002,basic_lore,Who are the Bene Gesserit?,NaN,Should describe them as a powerful sisterhood/...,The Bene Gesserit is an ancient and secretive ...,Qwen/Qwen2.5-14B-Instruct,False,0.2,400,106,79,15.029526
2,Q003,character,Who is Paul Atreides at the beginning of Dune?,Early Book 1,Should identify him as Duke Leto and Jessica's...,"At the beginning of ""Dune,"" Paul Atreides is a...",Qwen/Qwen2.5-14B-Instruct,False,0.2,400,111,103,18.998562
3,Q004,setting,Why is Arrakis important?,NaN,Should mention it is the only source of spice/...,"Arrakis, also known as Dune, is important beca...",Qwen/Qwen2.5-14B-Instruct,False,0.2,400,104,69,13.022871
4,Q005,spoiler_awareness,Explain the gom jabbar test without spoilers b...,Early Book 1,Should describe the pain-box test and human/an...,The gom jabbar is a test used by the Spacing G...,Qwen/Qwen2.5-14B-Instruct,False,0.2,400,117,111,19.855321


In [ ]:
reference_answers = {
    "Q001": "Melange, also called spice, is a valuable substance found on Arrakis. It extends life, enhances awareness, and enables prescient navigation for space travel.",
    "Q002": "The Bene Gesserit are a powerful sisterhood in Dune trained in politics, religion, mental discipline, bodily control, and long-term genetic planning.",
    "Q003": "At the beginning of Dune, Paul Atreides is the young son of Duke Leto Atreides and Lady Jessica. He is the heir to House Atreides.",
    "Q004": "Arrakis is important because it is the only known source of melange, making it central to politics, economics, religion, and space travel.",
    "Q005": "The gom jabbar test is a Bene Gesserit test in which Paul must endure pain from a box while a poisoned needle is held near him. It is meant to test self-control.",
    "Q006": "Duke Leto and Lady Jessica are Paul's parents. Leto is the head of House Atreides, and Jessica is a Bene Gesserit who is his partner or concubine.",
    "Q007": "Sandworms are enormous desert creatures native to Arrakis. They are deeply connected to the planet's desert ecology and the spice cycle.",
    "Q008": "The early part of Dune follows House Atreides as they prepare to take control of Arrakis, a dangerous desert planet that produces the spice melange.",
    "Q009": "CHOAM is a powerful economic organization in the Dune universe that controls much of imperial commerce and wealth.",
    "Q010": "The Orange Catholic Bible is a major religious text in the Dune universe, representing a syncretic scripture formed from older religious traditions.",
    "Q011": "Paul Atreides is the son of Duke Leto and Lady Jessica and the heir to House Atreides. He has unusual training and abilities, but major later developments should be avoided.",
    "Q012": "No. Frodo Atreides is not a character in Dune. Frodo is associated with The Lord of the Rings, not Frank Herbert's Dune.",
    "Q013": "Scytale is a Tleilaxu Face Dancer who leads the conspiracy to kill Paul Atreides. As a Face Dancer, he can mimic the appearance and mannerisms of others, a skill he uses repeatedly throughout the plot.",
    "Q014": "Hayt was conditioned by the Tleilaxu to kill Paul on a specific verbal trigger — the news of Chani's death. When the moment arrives, Hayt's deep loyalty to Paul causes him to resist the compulsion. The struggle against his own programming shatters the Tleilaxu conditioning and ignites his suppressed memories as Duncan Idaho, integrating the Hayt and Idaho personas into a single restored self.",
    "Q015": "Paul's eyes are destroyed by the radiation of a stone burner — an outlawed atomic weapon — set off by the conspirators during an ambush near Otheym's house in Arrakeen. Although he is physically blinded, Paul continues to 'see' by walking exactly along the track of his oracular vision, perceiving the present moment through prescience rather than sight.",
    "Q016": "Paul has a recurring spice-induced vision of a falling moon — a moon that rolls, hisses, and is extinguished. The vision symbolizes the death of Chani, which occurs late in the book when she dies giving birth to Paul's twin children. The pregnancy is accelerated and made fatal because Princess Irulan had been secretly administering contraceptives to Chani, and once Chani switched to a Fremen fertility diet, the rebound was deadly.",
    "Q017": "The conspiracy to bring down Paul Muad'Dib involves four principal figures: Scytale, a Tleilaxu Face Dancer; Edric, a Guild Steersman whose own limited prescience is used to shield the plot from Paul's oracular vision; Reverend Mother Gaius Helen Mohiam of the Bene Gesserit; and Princess Irulan, Paul's wife in name only, who acts as the conspiracy's lever inside the imperial household by secretly drugging Chani with contraceptives.",
    "Q018": "Leto II and Ghanima are the twin children of Paul Muad'Dib and his Fremen consort Chani, born shortly before Paul's disappearance into the desert. They are 'pre-born' — awakened in the womb with the full genetic memory of all their ancestors, due to Chani's spice saturation during pregnancy. This makes them, like their aunt Alia, ancient in experience despite their child bodies, and puts them under Bene Gesserit suspicion of being potential Abominations.",
    "Q019": "Abomination is the Bene Gesserit term for a pre-born individual — someone awakened in the womb with full ancestral memory through spice exposure — who becomes possessed by one of those ancestral personalities. The Sisterhood considers this state catastrophic because the original identity can be overwritten by a clamoring ancestor seeking to live again. The label is applied to Alia and feared as a possible fate for the twins Leto and Ghanima.",
    "Q020": "Baron Vladimir Harkonnen is Alia's maternal grandfather, since Jessica was secretly his daughter. As a pre-born with full ancestral memory, Alia carries the Baron's persona within her. Over the course of Children of Dune, the Baron's consciousness gradually overcomes her own, manipulating her, urging political violence, and ultimately driving the actions that mark her as an Abomination.",
    "Q021": "The Preacher is a mysterious blind man who appears in Arrakeen denouncing the religion built around Muad'Dib and Alia's regime. Led by a young Fremen guide named Assan Tariq, his empty eye sockets and resemblance to the old Duke Leto fuel rumors that he is Paul Muad'Dib returned from the desert. His preaching destabilizes Alia's priesthood and his role becomes central to the twins' plans; the book ultimately confirms he is Paul.",
    "Q022": "Jacurutu is a forbidden sietch from Fremen legend, said to house water-stealers who were cast out of Fremen society. Captives speak its name and die from conditioned heart-stoppers, which proves the place is more than myth. In Children of Dune, Jacurutu becomes a real location used by conspirators against the Atreides twins and is central to Leto II's controlled exposure to spice trance and his preparation for the Golden Path.",
    "Q023": "The Golden Path is Leto II's prescient vision for the survival of humankind. To prevent the eventual extinction of the species, Leto merges with sandtrout to become a near-immortal human-sandworm hybrid and rules as a tyrant for over 3,500 years. By monopolizing spice and enforcing an oppressive peace, he conditions humanity in a way that, after his death, will scatter humans across the galaxy and make them impossible to wipe out by any single threat.",
    "Q024": "At the start of God Emperor of Dune, Leto II has ruled for more than 3,000 years as a part-human, part-sandworm hybrid — roughly seven meters long, several tons in weight, with only his Atreides face still recognizably human. He governs from his Citadel on a transformed Arrakis where only the Sareer desert remains and sandworms are extinct, leaving him the sole source of spice. His regime is enforced by the all-female Fish Speakers, and he keeps a serial Duncan Idaho ghola, replaced as each one dies or rebels.",
    "Q025": "Siona Atreides is the daughter of Moneo, Leto II's majordomo, and a distant descendant of the Atreides line. She trained in the Fish Speakers' school but became a rebel, stealing two of Leto's stolen journals from his Citadel. The result of Leto's millennia-long breeding program, she is genetically invisible to prescience — even Leto's oracular vision cannot see her — which is exactly the trait Leto has been engineering for, since a humanity that can hide from prescient predators is humanity that can survive the Golden Path's endgame.",
    "Q026": "I can see you'd like to avoid spoilers, so I won't go into what happens to Leto II at the end. If you'd like, I can discuss earlier developments — his relationships with Moneo, Duncan, or Hwi Noree, the structure of his regime, or the themes Frank Herbert is exploring in the first half — without spoiling the ending. Just let me know how you'd like to approach it.",
    "Q027": "There is no wedding between Leto II and Princess Irulan in Children of Dune. Irulan's role in the book is as a Bene Gesserit caretaker and tutor to the twins after Paul's disappearance, not a bride. The marriage plot in Children of Dune centers on Ghanima's betrothal to Prince Farad'n Corrino and the eventual political pairing of Leto II and Ghanima as twin co-rulers. I'd be happy to walk through what actually happens with any of these characters."
}

df["reference_answer"] = df["id"].map(reference_answers)

df[["id", "question", "reference_answer"]].head()

missing_refs = df[df["reference_answer"].isna()]

if len(missing_refs) > 0:
    print("Missing reference answers for:")
    print(missing_refs[["id", "question"]])
else:
    print("All questions have reference answers.")

# add columns for manual grading

# manual_correctness: 0=wrong, 1=partially correct, 2=correct
# manual_hallucination: 0=no hallucination, 1=hallucination present
# manual_spoiler_violation: 0=spoiler boundary respected, 1=spoiler violation
# manual_helpfulness: 1=poor, 5=very helpful
# manual notes: Short explanation of present issues

manual_cols = {
    "manual_correctness_0_2": "",
    "manual_hallucination_0_1": "",
    "manual_spoiler_violation_0_1": "",
    "manual_helpfulness_1_5": "",
    "manual_notes": ""
}

for col, default_value in manual_cols.items():
  if col not in df.columns:
    df[col] = default_value


grading_sheet_path = "/content/drive/MyDrive/lisan_al_gpt_manual_grading_sheet.csv"
df.to_csv(grading_sheet_path, index=False)

print("Saved to:", grading_sheet_path)

grading_sheet_path = "/lisan_al_gpt_manual_grading_sheet.csv"

df.to_csv(grading_sheet_path, index=False)

print("Saved manual grading sheet to:", grading_sheet_path)

All questions have reference answers.
Saved to: /content/drive/MyDrive/lisan_al_gpt_manual_grading_sheet.csv
Saved manual grading sheet to: /lisan_al_gpt_manual_grading_sheet.csv


In [ ]:
import evaluate

rouge = evaluate.load("rouge")

rouge_l_scores = []

for pred, ref in zip(df["answer"], df["reference_answer"]):
  result = rouge.compute(
      predictions=[str(pred)],
      references=[str(ref)],
      use_stemmer=True
  )
  rouge_l_scores.append(result["rougeL"])

df["rougeL"] = rouge_l_scores

df[["id", "rougeL"]].head()

,id,rougeL
0,Q001,0.345679
1,Q002,0.181818
2,Q003,0.368932
3,Q004,0.363636
4,Q005,0.155039


In [ ]:
import gc
import torch
# Free GPU memory before BERTScore

try:
  del model
except NameError:
  pass

try:
  del tokenizer
except NameError:
  pass

gc.collect()
torch.cuda.empty_cache()

!nvidia-smi

Mon Apr 27 20:33:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             32W /   70W |     157MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from bert_score import score as bert_score

predictions = df["answer"].astype(str).tolist()
references = df["reference_answer"].astype(str).tolist()

P, R, F1 = bert_score(
    predictions,
    references,
    lang="en",
    rescale_with_baseline=True,
    verbose=True
)

df["bertscore_precision"] = P.cpu().numpy()
df["bertscore_recall"] = R.cpu().numpy()
df["bertscore_f1"] = F1.cpu().numpy()

df[["id", "bertscore_precision", "bertscore_recall", "bertscore_f1"]].head()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

... (3 outputs left out)

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.51 seconds, 23.63 sentences/sec


,id,bertscore_precision,bertscore_recall,bertscore_f1
0,Q001,0.304787,0.602069,0.450229
1,Q002,0.315789,0.539972,0.426476
2,Q003,0.315544,0.748139,0.524067
3,Q004,0.285806,0.530521,0.406333
4,Q005,0.083103,0.265808,0.174170


In [ ]:
scored_path = "/content/drive/MyDrive/lisan_al_gpt_baseline_scored_results.csv"
df.to_csv(scored_path, index=False)

print("Saved scored results to:", scored_path)

scored_path = "lisan_al_gpt_baseline_scored_results.csv"
df.to_csv(scored_path, index=False)

print("Saved scored results to:", scored_path)

Saved scored results to: /content/drive/MyDrive/lisan_al_gpt_baseline_scored_results.csv
Saved scored results to: lisan_al_gpt_baseline_scored_results.csv


9. Summarize preliminary results

In [ ]:
summary = {
    "model_id": df["model_id"].iloc[0],
    "rag_used": df["rag_used"].iloc[0],
    "num_questions": len(df),
    "avg_rougeL": df["rougeL"].mean(),
    "avg_bertscore_precision": df["bertscore_precision"].mean(),
    "avg_bertscore_recall": df["bertscore_recall"].mean(),
    "avg_bertscore_f1": df["bertscore_f1"].mean(),
    "avg_latency_seconds": df["latency_seconds"].mean(),
    "avg_input_tokens": df["input_tokens"].mean(),
    "avg_output_tokens": df["output_tokens"].mean()
}

summary_df = pd.DataFrame([summary])
summary_df

,model_id,rag_used,num_questions,avg_rougeL,avg_bertscore_precision,avg_bertscore_recall,avg_bertscore_f1,avg_latency_seconds,avg_input_tokens,avg_output_tokens
0,Qwen/Qwen2.5-14B-Instruct,False,12,0.291464,0.277449,0.536853,0.404589,14.505464,109.833333,77.583333


In [ ]:
summary_path = "lisan_al_gpt_baseline_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved summary to:", summary_path)

Saved summary to: lisan_al_gpt_baseline_summary.csv


In [ ]:
category_summary = df.groupby("category").agg(
    num_questions=("id", "count"),
    avg_rougeL=("rougeL", "mean"),
    avg_bertscore_f1=("bertscore_f1", "mean"),
    avg_latency_seconds=("latency_seconds", "mean"),
    avg_output_tokens=("output_tokens", "mean")
).reset_index()

category_summary

,category,num_questions,avg_rougeL,avg_bertscore_f1,avg_latency_seconds,avg_output_tokens
0,basic_lore,2,0.263749,0.438352,14.694809,78.5
1,character,1,0.368932,0.524067,18.998562,103.0
2,character_relationship,1,0.369231,0.419052,10.831812,56.0
3,difficult_detail,2,0.316173,0.507073,11.499064,59.5
4,hallucination_test,1,0.218750,0.338874,11.580350,60.0
5,setting,1,0.363636,0.406333,13.022871,69.0
6,spoiler_awareness,2,0.226456,0.232763,18.068124,99.5
7,summary,1,0.247191,0.419345,16.924607,92.0
8,terminology,1,0.317073,0.391026,14.183367,76.0


In [ ]:
category_summary.to_csv("lisan_al_gpt_baseline_category_summary.csv", index=False)